# 01 — Construction du réseau urbain

**Objectif** : construire un graphe routier unifié et enrichi pour une ville quelconque.

**Pipeline** :
1. Charger le réseau urbain via OSMnx
2. Charger les liaisons interurbaines depuis ROUTE500 (clip sur la zone tampon)
3. Enrichir chaque arc (capacité, vitesse libre, paramètres BPR)
4. Fusionner les deux sources
5. Construire l'objet `UrbanNetwork` final

**Ville de test** : Lyon

In [12]:
import osmnx as ox
import geopandas as gpd
import networkx as nx
import pandas as pd
import numpy as np
import folium
from shapely.geometry import box

from urban_optimizer.config import (
    RAW_DIR, PROCESSED_DIR,
    CAPACITY_BY_TYPE, FREE_SPEED_BY_TYPE,
    BPR_ALPHA, BPR_BETA,
    CRS_LAMBERT93, CRS_WGS84,
)
from urban_optimizer.utils.logging import get_logger

logger = get_logger("network_build")

CITY = "Lyon, France"
BUFFER_KM = 10

## Étape 1 — Réseau urbain via OSMnx

OSMnx télécharge le réseau routier d'OpenStreetMap. `network_type="drive"` filtre uniquement les routes praticables en voiture.

On garde tous les attributs OSM utiles : `highway` (type), `lanes` (nb voies), `maxspeed`, `oneway`.

In [13]:
(RAW_DIR / 'osmnx_cache').mkdir(parents=True, exist_ok=True)
ox.settings.use_cache = True
ox.settings.cache_folder = str(RAW_DIR / "osmnx_cache")
ox.settings.log_console = False

logger.info(f"Téléchargement OSM pour {CITY}")
G_osm = ox.graph_from_place(CITY, network_type="drive", simplify=True)

print(f"Nb nœuds OSM : {G_osm.number_of_nodes():,}")
print(f"Nb arcs OSM  : {G_osm.number_of_edges():,}")
print(f"CRS         : {G_osm.graph['crs']}")

13:38:42 | network_build | INFO | Téléchargement OSM pour Lyon, France
Nb nœuds OSM : 4,081
Nb arcs OSM  : 8,052
CRS         : epsg:4326


In [14]:
edges_gdf = ox.graph_to_gdfs(G_osm, nodes=False)
print("Colonnes disponibles :")
print(edges_gdf.columns.tolist())
print("\nRépartition par type de route :")
highway_counts = edges_gdf["highway"].astype(str).value_counts().head(15)
print(highway_counts)

Colonnes disponibles :
['osmid', 'highway', 'maxspeed', 'name', 'oneway', 'reversed', 'length', 'geometry', 'lanes', 'bridge', 'width', 'access', 'ref', 'tunnel', 'junction', 'est_width']

Répartition par type de route :
highway
residential                          3995
tertiary                             1102
primary                              1008
secondary                             780
unclassified                          650
living_street                         310
trunk_link                             54
['living_street', 'residential']       36
primary_link                           34
['residential', 'unclassified']        22
trunk                                  20
busway                                 14
secondary_link                          9
['living_street', 'unclassified']       6
tertiary_link                           6
Name: count, dtype: int64


## Étape 2 — Reprojection en Lambert-93

OSM est en WGS84 (degrés). Pour calculer les longueurs en mètres et faire des opérations spatiales correctes, on passe en Lambert-93.

In [15]:
G_osm_proj = ox.project_graph(G_osm, to_crs=f"EPSG:{CRS_LAMBERT93}")
edges_osm = ox.graph_to_gdfs(G_osm_proj, nodes=False)
nodes_osm = ox.graph_to_gdfs(G_osm_proj, edges=False)

print(f"CRS après reprojection : {edges_osm.crs}")
print(f"Étendue du réseau (Lambert-93) :")
print(f"  X: {edges_osm.total_bounds[0]:.0f} → {edges_osm.total_bounds[2]:.0f}")
print(f"  Y: {edges_osm.total_bounds[1]:.0f} → {edges_osm.total_bounds[3]:.0f}")

CRS après reprojection : EPSG:2154
Étendue du réseau (Lambert-93) :
  X: 837975 → 847562
  Y: 6514756 → 6524798


## Étape 3 — Enrichissement de chaque arc

Chaque arc doit porter :
- `length_m` : longueur en mètres
- `lanes` : nombre de voies (nettoyé)
- `free_speed_kmh` : vitesse libre estimée
- `capacity` : capacité en véh/h
- `t0_s` : temps de parcours libre
- `bpr_alpha`, `bpr_beta` : paramètres BPR

In [17]:
def normalize_highway(hw):
    """OSM renvoie parfois une liste de types. On garde le premier (le plus important)."""
    if isinstance(hw, list):
        return hw[0]
    return hw


def clean_lanes(lanes_value, default=1):
    """OSM `lanes` est en string, parfois liste/array numpy, parfois absent."""
    if lanes_value is None:
        return default
    try:
        if pd.isna(lanes_value):
            return default
    except (ValueError, TypeError):
        pass
    if isinstance(lanes_value, (list, np.ndarray)):
        try:
            return max(int(v) for v in lanes_value)
        except (ValueError, TypeError):
            return default
    try:
        return int(float(lanes_value))
    except (ValueError, TypeError):
        return default


def parse_maxspeed(ms):
    """`maxspeed` en string, parfois '30 mph', parfois '50 km/h', parfois absent."""
    if ms is None:
        return None
    try:
        if pd.isna(ms):
            return None
    except (ValueError, TypeError):
        pass
    if isinstance(ms, (list, np.ndarray)):
        ms = ms[0]
    s = str(ms).lower().strip()
    if "mph" in s:
        try:
            return int(float(s.replace("mph", "").strip()) * 1.609)
        except ValueError:
            return None
    try:
        return int(float(s.replace("km/h", "").strip()))
    except ValueError:
        return None


def enrich_edges(edges):
    """Ajoute toutes les colonnes nécessaires pour l'affectation."""
    edges = edges.copy()
    edges["highway_clean"] = edges["highway"].apply(normalize_highway)
    edges["lanes_clean"] = edges["lanes"].apply(clean_lanes) if "lanes" in edges else 1
    edges["length_m"] = edges.geometry.length

    edges["free_speed_kmh"] = edges["highway_clean"].map(FREE_SPEED_BY_TYPE).fillna(50)
    if "maxspeed" in edges.columns:
        observed = edges["maxspeed"].apply(parse_maxspeed)
        edges["free_speed_kmh"] = observed.fillna(edges["free_speed_kmh"])

    cap_per_lane = edges["highway_clean"].map(CAPACITY_BY_TYPE).fillna(600)
    edges["capacity"] = cap_per_lane * edges["lanes_clean"]

    edges["t0_s"] = edges["length_m"] / (edges["free_speed_kmh"] * 1000 / 3600)

    edges["bpr_alpha"] = BPR_ALPHA
    edges["bpr_beta"] = BPR_BETA

    return edges


edges_osm = enrich_edges(edges_osm)
edges_osm[["highway_clean", "lanes_clean", "length_m", "free_speed_kmh", "capacity", "t0_s"]].head(10)

highway_clean  lanes_clean    length_m  \
u        v          key                                          
143403   21714981   0     residential            1    7.261857   
         21718288   0     residential            1   24.997594   
         143408     0         primary            1  102.589938   
21714981 143403     0     residential            1    7.261857   
         8149051282 0         primary            2  125.410517   
         9226922647 0    unclassified            1   18.014943   
21718288 2490241911 0     residential            1  286.738877   
         21717931   0     residential            1  385.352487   
         143403     0     residential            1   24.997594   
143408   21718264   0     residential            2  309.106528   

                         free_speed_kmh  capacity       t0_s  
u        v          key                                       
143403   21714981   0              30.0     600.0   0.871423  
         21718288   0              30.0     600.0   2.999711  
         143408     0              50.0    1500.0   7.386476  
21714981 143403     0              30.0     600.0   0.871423  
         8149051282 0              50.0    3000.0   9.029557  
         9226922647 0              30.0     600.0   2.161793  
21718288 2490241911 0              30.0     600.0  34.408665  
         21717931   0              30.0     600.0  46.242298  
         143403     0              30.0     600.0   2.999711  
143408   21718264   0              30.0    1200.0  37.092783

In [18]:
print("Distribution des capacités (véh/h) :")
print(edges_osm["capacity"].describe())
print("\nDistribution des vitesses libres (km/h) :")
print(edges_osm["free_speed_kmh"].describe())

Distribution des capacités (véh/h) :
count    8052.000000
mean     1313.599106
std      1197.334892
min       600.000000
25%       600.000000
50%       600.000000
75%      1500.000000
max      7500.000000
Name: capacity, dtype: float64

Distribution des vitesses libres (km/h) :
count    8052.000000
mean       32.121833
std         7.456440
min        10.000000
25%        30.000000
50%        30.000000
75%        30.000000
max        90.000000
Name: free_speed_kmh, dtype: float64


## Étape 4 — Liaisons interurbaines via ROUTE500

On charge ROUTE500 et on garde uniquement les tronçons à proximité de la zone urbaine (bbox + 10 km), pour ajouter les autoroutes et nationales qui relient Lyon au reste.

In [19]:
ROUTE500_PATH = RAW_DIR / "ROUTE500_3-0__SHP_LAMB93_FXX_2021-11-03" / "ROUTE500" / "1_DONNEES_LIVRAISON_2022-01-00175" / "R500_3-0_SHP_LAMB93_FXX-ED211" / "RESEAU_ROUTIER" / "TRONCON_ROUTE.shp"

city_bounds = edges_osm.total_bounds
buffer_m = BUFFER_KM * 1000
bbox = box(
    city_bounds[0] - buffer_m,
    city_bounds[1] - buffer_m,
    city_bounds[2] + buffer_m,
    city_bounds[3] + buffer_m,
)
logger.info(f"Zone tampon : {bbox.bounds}")

r500 = gpd.read_file(ROUTE500_PATH, bbox=bbox, encoding="latin1")
print(f"Nb tronçons ROUTE500 dans la zone : {len(r500):,}")
print(f"CRS : {r500.crs}")
r500.head(3)

13:40:29 | network_build | INFO | Zone tampon : (827974.9042418925, 6504756.358494053, 857562.4094256293, 6534797.66381334)
Nb tronçons ROUTE500 dans la zone : 8,871
CRS : PROJCS["RGF93 Lambert 93",GEOGCS["RGF93 geographiques (dms)",DATUM["Reseau_Geodesique_Francais_1993_v1",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6171"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["IGNF","RGF93G"]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",46.5],PARAMETER["central_meridian",3],PARAMETER["standard_parallel_1",44],PARAMETER["standard_parallel_2",49],PARAMETER["false_easting",700000],PARAMETER["false_northing",6600000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["IGNF","LAMB93"]]


,ID_RTE500,VOCATION,NB_CHAUSSE,NB_VOIES,ETAT,ACCES,RES_VERT,SENS,NUM_ROUTE,RES_EUROPE,LONGUEUR,CLASS_ADM,geometry
0,BDCTRORO0000000315623286,Type autoroutier,2 chaussées,Sans objet,Revêtu,A péage,Appartient,Double sens,A89,E70,10.11,Autoroute,"LINESTRING (818784.8 6531912.6, 818793.6 65319..."
1,BDCTRORO0000000315623310,Type autoroutier,2 chaussées,Sans objet,Revêtu,Libre,Appartient,Double sens,A89,E70,0.22,Autoroute,"LINESTRING (829863.4 6528738.1, 829869.5 65287..."
2,BDCTRORO0000000315623284,Type autoroutier,2 chaussées,Sans objet,Revêtu,A péage,Appartient,Double sens,A89,E70,0.19,Autoroute,"LINESTRING (828266.3 6529578.9, 828315.5 65295..."


In [20]:
VOCATION_TO_HIGHWAY = {
    "Type autoroutier": "motorway",
    "Liaison principale": "trunk",
    "Liaison régionale": "primary",
    "Liaison locale": "secondary",
    "Bretelle": "motorway_link",
}

VOCATION_LANES = {
    "Type autoroutier": 2,
    "Liaison principale": 1,
    "Liaison régionale": 1,
    "Liaison locale": 1,
    "Bretelle": 1,
}


def r500_to_standard(r500_gdf):
    """Convertit ROUTE500 vers le même schéma que les arcs OSM enrichis."""
    out = gpd.GeoDataFrame(geometry=r500_gdf.geometry, crs=r500_gdf.crs)
    out["highway_clean"] = r500_gdf["VOCATION"].map(VOCATION_TO_HIGHWAY).fillna("secondary")
    out["lanes_clean"] = r500_gdf["NB_CHAUSSE"].apply(
        lambda x: 2 if isinstance(x, str) and "2" in x else 1
    )
    out["lanes_clean"] = out["lanes_clean"] * r500_gdf["VOCATION"].map(VOCATION_LANES).fillna(1)
    out["length_m"] = out.geometry.length
    out["free_speed_kmh"] = out["highway_clean"].map(FREE_SPEED_BY_TYPE).fillna(70)
    out["capacity"] = out["highway_clean"].map(CAPACITY_BY_TYPE).fillna(900) * out["lanes_clean"]
    out["t0_s"] = out["length_m"] / (out["free_speed_kmh"] * 1000 / 3600)
    out["bpr_alpha"] = BPR_ALPHA
    out["bpr_beta"] = BPR_BETA
    out["source"] = "route500"
    out["r500_id"] = r500_gdf["ID_RTE500"].values
    out["oneway"] = r500_gdf["SENS"].apply(
        lambda s: isinstance(s, str) and ("direct" in s.lower() or "inverse" in s.lower())
    )
    return out


r500_clean = r500_to_standard(r500)
edges_osm["source"] = "osm"
r500_clean.head()

,geometry,highway_clean,lanes_clean,length_m,free_speed_kmh,capacity,t0_s,bpr_alpha,bpr_beta,source,r500_id,oneway
0,"LINESTRING (818784.8 6531912.6, 818793.6 65319...",motorway,4,10108.067096,130.0,8000.0,279.915704,0.15,4.0,route500,BDCTRORO0000000315623286,False
1,"LINESTRING (829863.4 6528738.1, 829869.5 65287...",motorway,4,223.531773,130.0,8000.0,6.190111,0.15,4.0,route500,BDCTRORO0000000315623310,False
2,"LINESTRING (828266.3 6529578.9, 828315.5 65295...",motorway,4,188.204861,130.0,8000.0,5.211827,0.15,4.0,route500,BDCTRORO0000000315623284,False
3,"LINESTRING (828453.8 6529566.8, 828615.3 65295...",motorway,4,486.508465,130.0,8000.0,13.472542,0.15,4.0,route500,BDCTRORO0000000315623278,False
4,"LINESTRING (829915.5 6528521, 829952.7 6528447...",motorway,4,253.381622,130.0,8000.0,7.016722,0.15,4.0,route500,BDCTRORO0000000315623273,False


## Étape 5 — Éviter le double comptage urbain × ROUTE500

À l'intérieur de Lyon, OSM couvre déjà tout. ROUTE500 doublonnerait inutilement. Stratégie simple : on retire de ROUTE500 les tronçons dont le centroïde est dans l'enveloppe urbaine OSM.

In [21]:
osm_envelope = edges_osm.union_all().buffer(200).convex_hull
r500_centroids_in = r500_clean.geometry.centroid.within(osm_envelope)
r500_outside = r500_clean[~r500_centroids_in].copy()

print(f"ROUTE500 conservé (hors zone urbaine OSM) : {len(r500_outside):,}")
print(f"ROUTE500 retiré (déjà couvert par OSM)   : {r500_centroids_in.sum():,}")

ROUTE500 conservé (hors zone urbaine OSM) : 7,020
ROUTE500 retiré (déjà couvert par OSM)   : 1,851


## Étape 6 — Construction du graphe igraph unifié

On rassemble tout dans un objet `UrbanNetwork` qui contient :
- le graphe `igraph` orienté
- l'index `node_id → (x, y)` en Lambert-93
- le GeoDataFrame des arcs pour la viz

C'est cet objet qu'on passera aux briques suivantes.

In [22]:
import igraph as ig
from dataclasses import dataclass


@dataclass
class UrbanNetwork:
    graph: ig.Graph
    nodes_xy: dict
    edges_gdf: gpd.GeoDataFrame
    crs: int

    def summary(self):
        print(f"Graphe : {self.graph.vcount():,} nœuds, {self.graph.ecount():,} arcs")
        print(f"CRS    : EPSG:{self.crs}")
        print(f"Sources: {self.edges_gdf['source'].value_counts().to_dict()}")


def build_unified_network(edges_osm_gdf, nodes_osm_gdf, edges_r500, crs=CRS_LAMBERT93):
    """Construit le graphe orienté igraph à partir des deux sources enrichies.

    Args:
        edges_osm_gdf  : GeoDataFrame des arcs OSM (avec colonnes enrichies)
        nodes_osm_gdf  : GeoDataFrame des nœuds OSM (index = osmid, geometry = Point)
        edges_r500     : GeoDataFrame des arcs ROUTE500 filtrés et enrichis
        crs            : code EPSG du CRS de travail (Lambert-93 par défaut)

    Returns:
        UrbanNetwork avec graphe igraph, index des nœuds, et GeoDataFrame des arcs.
    """

    # --- Nœuds OSM : coordonnées arrondies → index entier ---
    osm_id_to_coord = {
        nid: (round(row.geometry.x, 1), round(row.geometry.y, 1))
        for nid, row in nodes_osm_gdf.iterrows()
    }

    all_coords = set(osm_id_to_coord.values())

    # --- Extrémités des arcs ROUTE500 depuis la géométrie ---
    def endpoints_from_geom(geom):
        coords = list(geom.coords)
        return (
            (round(coords[0][0], 1), round(coords[0][1], 1)),
            (round(coords[-1][0], 1), round(coords[-1][1], 1)),
        )

    r500 = edges_r500.copy()
    eps = r500.geometry.apply(endpoints_from_geom)
    r500["u_xy"] = eps.apply(lambda x: x[0])
    r500["v_xy"] = eps.apply(lambda x: x[1])

    all_coords.update(r500["u_xy"])
    all_coords.update(r500["v_xy"])

    coord_to_idx = {c: i for i, c in enumerate(sorted(all_coords))}

    # --- Construction des listes d'arcs et attributs ---
    edges_list = []
    attrs = {
        "length_m": [], "free_speed_kmh": [], "capacity": [],
        "t0_s": [], "bpr_alpha": [], "bpr_beta": [], "highway": [], "source": [],
    }
    geoms = []

    osm_reset = edges_osm_gdf.reset_index()
    for _, row in osm_reset.iterrows():
        u_c = osm_id_to_coord.get(row["u"])
        v_c = osm_id_to_coord.get(row["v"])
        if u_c is None or v_c is None:
            continue
        edges_list.append((coord_to_idx[u_c], coord_to_idx[v_c]))
        for k in ["length_m", "free_speed_kmh", "capacity", "t0_s", "bpr_alpha", "bpr_beta"]:
            attrs[k].append(float(row[k]))
        attrs["highway"].append(str(row["highway_clean"]))
        attrs["source"].append("osm")
        geoms.append(row.geometry)

    for _, row in r500.iterrows():
        u_idx = coord_to_idx[row["u_xy"]]
        v_idx = coord_to_idx[row["v_xy"]]
        directions = [(u_idx, v_idx)] if row.get("oneway", False) else [(u_idx, v_idx), (v_idx, u_idx)]
        for src_n, tgt_n in directions:
            edges_list.append((src_n, tgt_n))
            for k in ["length_m", "free_speed_kmh", "capacity", "t0_s", "bpr_alpha", "bpr_beta"]:
                attrs[k].append(float(row[k]))
            attrs["highway"].append(str(row["highway_clean"]))
            attrs["source"].append("route500")
            geoms.append(row.geometry)

    # --- Graphe igraph ---
    g = ig.Graph(n=len(coord_to_idx), edges=edges_list, directed=True)
    for k, vals in attrs.items():
        g.es[k] = vals

    nodes_xy = {i: c for c, i in coord_to_idx.items()}

    edges_gdf_out = gpd.GeoDataFrame(
        {k: attrs[k] for k in attrs},
        geometry=geoms,
        crs=f"EPSG:{crs}",
    )

    return UrbanNetwork(graph=g, nodes_xy=nodes_xy, edges_gdf=edges_gdf_out, crs=crs)


# Appel avec nodes_osm passé explicitement
net = build_unified_network(edges_osm, nodes_osm, r500_outside)
net.summary()

Graphe : 8,837 nœuds, 21,859 arcs
CRS    : EPSG:2154
Sources: {'route500': 13807, 'osm': 8052}


## Étape 7 — Tests de cohérence

Vérifications de sanité avant de passer à la suite.

In [23]:
g = net.graph

components = g.connected_components(mode="weak")
largest = max(components.sizes())
print(f"Composantes faiblement connexes : {len(components)}")
print(f"Taille de la plus grande : {largest:,} nœuds ({largest/g.vcount()*100:.1f}% du total)")

neg_lengths = (np.array(g.es["length_m"]) <= 0).sum()
print(f"\nArcs de longueur nulle ou négative : {neg_lengths}")

neg_cap = (np.array(g.es["capacity"]) <= 0).sum()
print(f"Arcs de capacité nulle ou négative : {neg_cap}")

neg_t0 = (np.array(g.es["t0_s"]) <= 0).sum()
print(f"Arcs de temps nul ou négatif : {neg_t0}")

Composantes faiblement connexes : 4
Taille de la plus grande : 4,751 nœuds (53.8% du total)

Arcs de longueur nulle ou négative : 0
Arcs de capacité nulle ou négative : 0
Arcs de temps nul ou négatif : 0


In [24]:
import time

rng = np.random.default_rng(42)
n_tests = 100
sources = rng.choice(g.vcount(), n_tests)
targets = rng.choice(g.vcount(), n_tests)

t0 = time.time()
for s, t in zip(sources, targets):
    _ = g.get_shortest_paths(s, to=t, weights="t0_s", mode="out")
elapsed = time.time() - t0
print(f"100 plus courts chemins en {elapsed:.2f}s ({elapsed/n_tests*1000:.1f} ms par requête)")

/var/folders/2m/0n4htc9j1njf5ss1d19h5d900000gn/T/ipykernel_92194/4156181498.py:10: RuntimeWarning: Couldn't reach some vertices. Location: src/paths/dijkstra.c:555
  _ = g.get_shortest_paths(s, to=t, weights="t0_s", mode="out")


100 plus courts chemins en 0.28s (2.8 ms par requête)


## Étape 8 — Visualisation

Carte folium colorée par capacité, pour vérifier que la fusion OSM+ROUTE500 est cohérente.

In [25]:
edges_wgs84 = net.edges_gdf.to_crs(CRS_WGS84)

sample = edges_wgs84.sample(min(3000, len(edges_wgs84)), random_state=42)

centroid = edges_wgs84.geometry.union_all().centroid
m = folium.Map(location=[centroid.y, centroid.x], zoom_start=11, tiles="CartoDB positron")

def cap_color(cap):
    if cap >= 3000:
        return "#c0392b"
    if cap >= 1500:
        return "#e67e22"
    if cap >= 900:
        return "#f1c40f"
    return "#7f8c8d"

for _, row in sample.iterrows():
    if row.geometry is None:
        continue
    if row.geometry.geom_type == "LineString":
        coords = [(y, x) for x, y in row.geometry.coords]
        folium.PolyLine(
            locations=coords,
            color=cap_color(row["capacity"]),
            weight=1.5,
            opacity=0.7,
            tooltip=f"{row['highway']} | cap={row['capacity']:.0f} v/h | source={row['source']}",
        ).add_to(m)

m

## Étape 9 — Sauvegarde

On sauvegarde le réseau enrichi pour les briques suivantes.

In [26]:
import pickle

out_path = PROCESSED_DIR / "lyon_network.pkl"
with open(out_path, "wb") as f:
    pickle.dump(net, f)

print(f"Réseau sauvegardé : {out_path}")
print(f"Taille : {out_path.stat().st_size / 1e6:.1f} MB")

Réseau sauvegardé : /Users/enzo/Desktop/B3/ML for business Project/Applied-Machine-Learning-for-business/data/processed/lyon_network.pkl
Taille : 5.4 MB


## Prochaine étape

Le notebook est validé. Une fois que la logique tourne, on extrait :
- `urban_optimizer/network/osm_loader.py` (étape 1-3)
- `urban_optimizer/network/route500_loader.py` (étape 4-5)
- `urban_optimizer/network/merger.py` (étape 6)
- `urban_optimizer/network/urban_network.py` (la dataclass)

Avec en façade publique : `build_network(city_name)` qui fait tout.